<a href="https://colab.research.google.com/github/Rahulthebeast619/Document-Similarity-/blob/main/week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF Document Similarity Using a Vector Space Model

Revised to follow [rashydaii/Tech400](https://github.com/rashydaii/Tech400): `pypdf`, `TfidfVectorizer(stop_words="english", lowercase=True, max_df=0.95, min_df=1)`, and cosine similarity.

**Run instructions:** Extract `Tech400_Revised_Project.zip`, open this notebook in its `Tech400_Revised` folder, and run all cells in order. The folder must contain `documents/` and `extracted_text/`. Install the packages in `requirements.txt` using your notebook kernel. The supplied verified OCR cache means Tesseract is not needed for this dataset unless you delete that cache. A fresh OCR run requires the external Tesseract program with English language data.

The original `article3.pdf` is image-only. Its original zero vector was an extraction failure, not proof of no similarity. This version adds OCR and rejects zero vectors. Saved outputs below come from executing this notebook's Python cells. The revised files have not been uploaded to the repository.


## 1. Imports and paths

In [ ]:
"""PDF document similarity, following rashydaii/Tech400's TF-IDF workflow.
Run: python vector_space_model.py
The five original PDFs belong in documents/. Image-only PDFs use Tesseract OCR.
"""
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import tempfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DOCUMENT_FOLDER = ROOT / 'documents'
OUTPUT_FOLDER = ROOT / 'results'
TEXT_FOLDER = ROOT / 'extracted_text'




## 2. OCR fallback and cache validation

In [ ]:
def ocr_pdf(file_path):
    """Recover an image-only PDF; keep cached text tied to the PDF hash."""
    pdf_hash = hashlib.sha256(file_path.read_bytes()).hexdigest()
    cache = TEXT_FOLDER / (file_path.stem + '.ocr.txt')
    meta = TEXT_FOLDER / (file_path.stem + '.ocr.json')
    if cache.exists() and meta.exists():
        record = json.loads(meta.read_text())
        text = cache.read_text(encoding='utf-8')
        text_hash = hashlib.sha256(text.encode()).hexdigest()
        if record.get('pdf_sha256') == pdf_hash and record.get('text_sha256') == text_hash:
            return text, 'OCR cached'
    if not shutil.which('tesseract'):
        raise RuntimeError(f'{file_path.name}: no extractable text. Install Tesseract OCR '
                           'or retain the supplied verified OCR cache.')
    import fitz  # PyMuPDF renders pages for Tesseract.

    def read_page(page_number):
        # A separate PDF handle per worker avoids sharing PyMuPDF objects.
        with fitz.open(file_path) as pdf, tempfile.TemporaryDirectory() as temp:
            picture = Path(temp) / 'page.png'
            pdf[page_number].get_pixmap(dpi=200).save(picture)
            result = subprocess.run(
                ['tesseract', str(picture), 'stdout', '-l', 'eng', '--psm', '3'],
                check=True, capture_output=True, text=True)
            return result.stdout

    with fitz.open(file_path) as pdf:
        page_count = len(pdf)
    print(f'OCR: {file_path.name} ({page_count} pages)', flush=True)
    with ThreadPoolExecutor(max_workers=4) as executor:
        pages = list(executor.map(read_page, range(page_count)))
    text = '\n\f\n'.join(pages)
    if not text.strip():
        raise ValueError(f'OCR produced no text: {file_path.name}')
    cache.write_text(text, encoding='utf-8')
    record = dict(pdf_sha256=pdf_hash,
                  text_sha256=hashlib.sha256(text.encode()).hexdigest(),
                  engine='Tesseract', language='eng', dpi=200, psm=3, pages=page_count)
    meta.write_text(json.dumps(record, indent=2))
    return text, 'OCR generated'




## 3. PDF extraction

In [ ]:
def read_documents():
    # 1. PDF DOCUMENT FOLDER AND TEXT EXTRACTION
    files = sorted(DOCUMENT_FOLDER.glob('*.pdf'))
    if not 5 <= len(files) <= 10:
        raise ValueError('Place 5 to 10 PDF documents in the documents folder.')
    TEXT_FOLDER.mkdir(exist_ok=True)
    documents, document_names, extraction = [], [], []
    for file_path in files:
        reader = PdfReader(file_path)
        text = ''.join((page.extract_text() or '') + ' ' for page in reader.pages)
        method = 'pypdf'
        if not text.strip():
            text, method = ocr_pdf(file_path)
        if not text.strip():
            raise ValueError(f'Empty document: {file_path.name}')
        (TEXT_FOLDER / (file_path.stem + '.txt')).write_text(text, encoding='utf-8')
        documents.append(text)
        document_names.append(file_path.name)
        extraction.append(dict(document=file_path.name, pages=len(reader.pages),
                               characters=len(text), method=method))
        print(f'Reading: {file_path.name} | {len(reader.pages)} pages | '
              f'{len(text)} characters | {method}', flush=True)
    return documents, document_names, extraction




## 4. TF-IDF, cosine similarity and CSV export

In [ ]:
def main():
    OUTPUT_FOLDER.mkdir(exist_ok=True)
    documents, document_names, extraction = read_documents()
    print('\nCREATING TF-IDF VECTORS')
    # 2. SAME TF-IDF SETTINGS AS THE ORIGINAL REPOSITORY
    vectorizer = TfidfVectorizer(
        stop_words='english',
        lowercase=True,
        max_df=0.95,
        min_df=1
    )
    tfidf_matrix = vectorizer.fit_transform(documents)
    zero_rows = np.asarray(tfidf_matrix.getnnz(axis=1)).ravel() == 0
    if zero_rows.any():
        empty = [name for name, flag in zip(document_names, zero_rows) if flag]
        raise ValueError(f'Documents have no retained terms: {empty}')
    print('TF-IDF matrix shape:', tfidf_matrix.shape)
    print('Number of unique terms:', len(vectorizer.get_feature_names_out()))

    # 3. COSINE SIMILARITY AND MATRIX
    similarity_matrix = cosine_similarity(tfidf_matrix)
    assert np.allclose(similarity_matrix, similarity_matrix.T)
    assert np.allclose(np.diag(similarity_matrix), 1.0)
    assert similarity_matrix.min() >= -1e-12
    assert similarity_matrix.max() <= 1 + 1e-12
    similarity_df = pd.DataFrame(similarity_matrix,
                                index=document_names, columns=document_names)
    print('\nCOSINE SIMILARITY MATRIX')
    print(similarity_df.round(4).to_string())
    # Save full precision; round only the displayed values.
    similarity_df.to_csv(OUTPUT_FOLDER / 'similarity_matrix.csv')

    # 4. RANK EVERY UNIQUE DOCUMENT PAIR
    pairs = []
    for i in range(len(document_names)):
        for j in range(i + 1, len(document_names)):
            pairs.append({'Document 1': document_names[i],
                          'Document 2': document_names[j],
                          'Similarity Score': float(similarity_matrix[i, j])})
    pairs_df = pd.DataFrame(pairs).sort_values('Similarity Score', ascending=False)
    pairs_df.to_csv(OUTPUT_FOLDER / 'document_similarity_results.csv', index=False)
    print('\nDOCUMENT SIMILARITY RANKING')
    print(pairs_df.round(4).to_string(index=False))
    print('\nMOST SIMILAR DOCUMENTS\n', pairs_df.iloc[0].round(4).to_string())
    print('\nLEAST SIMILAR DOCUMENTS\n', pairs_df.iloc[-1].round(4).to_string())

    # 5. TOP TERMS AND EXTRACTION RECORD
    feature_names = vectorizer.get_feature_names_out()
    top_terms = {}
    print('\nTOP TERMS IN EACH DOCUMENT')
    for i, name in enumerate(document_names):
        row = tfidf_matrix[i].toarray().ravel()
        top_terms[name] = [feature_names[k] for k in row.argsort()[-5:][::-1] if row[k] > 0]
        print(name + ': ' + ', '.join(top_terms[name]))
    summary = dict(documents=len(documents), terms=tfidf_matrix.shape[1],
                   pairs=len(pairs), mean=float(pairs_df['Similarity Score'].mean()),
                   highest=pairs_df.iloc[0].to_dict(), lowest=pairs_df.iloc[-1].to_dict(),
                   extraction=extraction, top_terms=top_terms)
    (OUTPUT_FOLDER / 'summary.json').write_text(json.dumps(summary, indent=2))
    print('\nPASS: nonzero vectors, symmetry, unit diagonal and score bounds.')
    print('PROGRAM COMPLETED SUCCESSFULLY')
    return summary




## 5. Run the analysis
This reads all five PDF files and prints the matrix, ranking, top terms and validation results. Full-precision CSV files are saved in `results/`.

In [ ]:
summary = main()


Reading: article1.pdf | 2 pages | 3049 characters | pypdf
Reading: article2.pdf | 32 pages | 89410 characters | pypdf
Reading: article3.pdf | 49 pages | 157819 characters | OCR cached
Reading: article4.pdf | 9 pages | 20369 characters | pypdf
Reading: article5.pdf | 48 pages | 106494 characters | pypdf

CREATING TF-IDF VECTORS
TF-IDF matrix shape: (5, 5636)
Number of unique terms: 5636

COSINE SIMILARITY MATRIX
              article1.pdf  article2.pdf  article3.pdf  article4.pdf  article5.pdf
article1.pdf        1.0000        0.4127        0.0274        0.0121        0.0577
article2.pdf        0.4127        1.0000        0.1853        0.1930        0.2222
article3.pdf        0.0274        0.1853        1.0000        0.1062        0.1075
article4.pdf        0.0121        0.1930        0.1062        1.0000        0.1363
article5.pdf        0.0577        0.2222        0.1075        0.1363        1.0000

DOCUMENT SIMILARITY RANKING
  Document 1   Document 2  Similarity Score
article1.pdf a

## Interpretation

The documents are: article1 (OECD AI principles), article2 (OECD government use of AI), article3 (UNESCO generative AI), article4 (WHO health ethics executive summary), and article5 (NIST AI RMF).

There are five nonzero document vectors and ten unique pairs. The highest similarity is **0.4127** for article1–article2; the lowest is **0.0121** for article1–article4. Scores changed after OCR because both the recovered text and corpus-wide IDF/document-frequency filtering affect the model. With five documents, `max_df=0.95` excludes any term present in all five.

OCR is imperfect: for example, `AI`/`GenAI` can be read as `Al`/`GenAl`. Titles, references and repeated page text remain in the model to preserve the original extraction approach. Cosine similarity measures lexical overlap, not semantic accuracy.



